# Check frozen core (AGFN)

Analogous to RxnFlow's `notebooks/06_check_frozen_core.ipynb`. We seed generation with a fixed
molecule whose **core is held immutable**, grow molecules from it, and verify the core stays
intact — and stays **identifiable** — throughout generation.

**How AGFN freezes the core.** AGFN builds molecules atom-by-atom on a `networkx` graph, assigning
each new atom the label `max(nodes)+1` and never renumbering. So the seed's atoms keep node labels
`0..N-1` for the whole trajectory. `build_frozen_seed_graph` stamps `g.graph['frozen_seed_size']`
and `g.graph['frozen_growth_sites']`; `graph_to_Data` masks any action that would edit the core, and
`GraphBuildingEnv.step` asserts the same as a safety net. This part is robust — Checks A and B pass.

**Why we tag the core (the fix).** The *graph* keeps stable labels, but `graph_to_mol` ends with a
canonical-SMILES round-trip (`MolFromSmiles(MolToSmiles(...))`) that **reorders RDKit atoms**, so an
atom's RDKit index in the emitted molecule is *not* its node label — and the reordering shifts as the
molecule grows. Highlighting `range(N)` therefore shades the wrong atoms. Mirroring RxnFlow, we make
core identity *chemical, not positional*: `graph_to_mol(g, tag_frozen=True)` stamps each core atom
with a reserved atom-map number `FROZEN_MAP_BASE + label` that survives the round-trip, and
`frozen_core_atoms(mol)` locates the core by that tag instead of by index.

Three checks per generated molecule:
- **Check A (exact):** the subgraph induced on labels `0..N-1` equals the seed (atoms, bonds, attrs).
- **Check B (chemistry):** `generated_mol.HasSubstructMatch(seed_mol)`, where `seed_mol` is the
  frozen core as the env represents it (`graph_to_mol(seed_graph)`), not a re-parse of the raw SMILES.
- **Check C (identity, across consecutive rollouts):** the atoms we *highlight* — located by map
  number — are exactly one of the seed's substructure matches, i.e. the red region equals the seed.

In [ ]:
import sys
from pathlib import Path

# The only boilerplate: make `import nbtools` resolve, then bootstrap the repo (chdir to root +
# put src/ on sys.path). Importing nbtools loads py3Dmol first, so rdkit.Chem.Draw stays safe.
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "nbtools").is_dir():
        sys.path.insert(0, str(_c)); break
    if (_c / "notebooks" / "nbtools").is_dir():
        sys.path.insert(0, str(_c / "notebooks")); break
import nbtools
from nbtools import sampling, frozen_core, render2d
# chdir=False: this notebook uses an absolute checkpoint path and no repo-relative configs, so it
# stays in notebooks/ (where its saved check_frozen_core_grid.png belongs); we only need src/ on path.
REPO_ROOT = nbtools.setup_repo(chdir=False)
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
print("repo root:", REPO_ROOT)

## Configuration

`SEED_SMILES` must use only atoms in the checkpoint's atom set (`['Br','C','Cl','F','I','N','O','S']`) and be smaller than `max_nodes` (45) to leave room to
grow. `GROWTH_ATOMS` are the 0-based seed atom indices where new atoms may attach; every other
core atom is frozen. Set `GROWTH_ATOMS = None` to freeze the whole core but allow growth at any
atom, or swap to `FROZEN_ATOMS = [...]` to name the atoms to freeze instead.

In [ ]:
CKPT = "/home/aid/dkoh/log/generative/trash/braf_RTB_Denovo_20260527_174606/model_state_braf_RTB_Denovo_20260527_174606_0.pt"
SEED_SMILES = "COC1=CC=C(F)C(F)=C1C(C=C2)=CN3C2=NC(NC([C@@H]4[C@@H](C)C4)=O)=C3"   # benzene; swap for any seed over the model's atom set
GROWTH_ATOMS = [22]          # grow only off atom 0; the other 5 ring atoms stay frozen
FROZEN_ATOMS = None         # alternative: name atoms to freeze instead (mutually exclusive)
N_SAMPLES = 128

In [ ]:
# 2D structure with each atom's index annotated -- read off the numbers you want to freeze.
render2d.draw_mol_with_indices(Chem.MolFromSmiles(SEED_SMILES), size=(360, 280))

In [ ]:
sampler = sampling.build_sampler(
    CKPT, initial_scaffold=SEED_SMILES,
    allowed_growth_atoms=GROWTH_ATOMS, frozen_atoms=FROZEN_ATOMS,
)
N = sampler.seed_graph.graph["frozen_seed_size"]
growth_sites = sampler.seed_graph.graph["frozen_growth_sites"]
print(f"seed = {SEED_SMILES!r}: {N} core atoms, growth sites = {sorted(growth_sites)}")

# Reference molecule for the chemistry-level checks (B and C): the frozen core *as the env
# represents it* (graph_to_mol of the seed graph), NOT a re-parse of the raw SMILES. The env drops
# parse artifacts (e.g. a radical at an under-valent growth atom) that never appear in generated
# molecules, so it is the correct, artifact-proof reference.
seed_mol = sampler.ctx.graph_to_mol(sampler.seed_graph)
env_seed_canon = Chem.MolToSmiles(seed_mol)
raw_seed_canon = Chem.MolToSmiles(Chem.MolFromSmiles(SEED_SMILES))
print(f"frozen core (env reference): {env_seed_canon}")
if env_seed_canon != raw_seed_canon:
    print(f"note: raw SEED_SMILES canonicalizes to {raw_seed_canon!r}, which differs from the frozen")
    print("      core above -- the env normalized away a parse artifact at an under-valent growth atom.")

## Sample

Call `sample_from_model` directly (rather than `sample_smiles`) so we can inspect each final
trajectory **graph**, which Check A needs.

In [ ]:
# Sample N_SAMPLES trajectories with the seed pinned; keep valid ones (with fwd_logprob). Convert
# final graphs to plain mols (Check B) and core-tagged mols (Check C / visualization).
valid = sampling.sample_trajectories(sampler, N_SAMPLES, seed_graph=sampler.seed_graph,
                                     require_logprob=True)
graphs = [t["traj"][-1][0] for t in valid]
mols = [sampler.ctx.graph_to_mol(g) for g in graphs]
tagged_mols = [sampler.ctx.graph_to_mol(g, tag_frozen=True) for g in graphs]
grew = sum(len(g.nodes) > N for g in graphs)
print(f"valid trajectories: {len(valid)}/{N_SAMPLES}   |   grew beyond seed: {grew}/{len(graphs)}")

## Check A — exact core subgraph

Since seed atoms keep labels `0..N-1`, compare the induced subgraph on those labels (atoms,
bonds, and their attribute dicts) against the seed. This is the strict invariant the masks +
step guards enforce.

In [ ]:
a_pass = sum(frozen_core.check_core_intact_graph(g, sampler.seed_graph, N) for g in graphs)
print(f"Check A (exact core subgraph): {a_pass}/{len(graphs)} = {100.0 * a_pass / max(len(graphs), 1):.1f}%")

## Check B — RDKit substructure match

A chemistry-level cross-check: the generated molecule must contain the seed as a substructure.

The reference `seed_mol` is the frozen core **as the env represents it** (`graph_to_mol(seed_graph)`,
defined in the sampler cell), not a re-parse of the raw `SEED_SMILES`. The raw SMILES can carry RDKit
parse artifacts — e.g. an under-valent growth atom written as `[C@@H]` becomes a **carbon radical** —
that `mol_to_graph` intentionally drops and that therefore never appear in generated molecules.
Matching against such an artifact gives false failures (RDKit's default matcher requires the query's
radical count to be matched), so we compare against exactly what was frozen.

In [ ]:
# seed_mol is the env's rendering of the frozen core (defined in the sampler cell), not a re-parse
# of the raw SMILES -- see that cell for why (input SMILES can carry parse artifacts the env drops).
b_pass = frozen_core.check_substructure(mols, seed_mol)
print(f"Check B (substructure match): {b_pass}/{len(mols)} = {100.0 * b_pass / max(len(mols), 1):.1f}%")

## Check C — labeled core == seed across consecutive rollouts

The decisive check for this notebook's purpose. We roll out **several independent sampling
batches** and, for each generated molecule, locate the core the way the visualization does — by
the reserved atom-map tag, via `frozen_core_atoms` — then confirm that set of atoms is **exactly
one of the seed's substructure matches**. That ties the highlighted (red) region to the seed
chemically, so a pass means the red atoms in the pictures below really are the seed molecule,
not an index that happens to line up. We expect 100% in every round.

As in Check B, the `seed_mol` reference is the env's frozen core (`graph_to_mol(seed_graph)`), so the
match is against exactly what was frozen rather than a re-parse of the raw SMILES.

In [ ]:
# Re-sample several independent batches and confirm, for every grown molecule, that the atoms we
# would HIGHLIGHT -- located by reserved map number, not by index -- are exactly the seed.
ROUNDS = 3
res_c = frozen_core.check_labeled_core(sampler, N_SAMPLES, ROUNDS, seed_mol, N)
total, c_count, c_redseed = res_c["total"], res_c["c_count"], res_c["c_redseed"]
success_modes, failure_modes = res_c["success"], res_c["failure"]
assert total > 0 and c_count == total == c_redseed, "labeled core does not match the seed somewhere!"
print("\nPASS: the labeled (red) core equals the seed molecule in every sampled molecule.")

## Summary

In [ ]:
print(f"valid              : {len(valid)}/{N_SAMPLES}")
print(f"grew beyond seed   : {grew}/{len(graphs)}")
print(f"Check A (exact)    : {a_pass}/{len(graphs)}")
print(f"Check B (substr)   : {b_pass}/{len(mols)}")
print(f"Check C (red==seed): {c_redseed}/{total}  (across {ROUNDS} consecutive rollouts)")
assert len(graphs) > 0, "no valid molecules generated"
assert a_pass == len(graphs), "frozen core was edited in at least one molecule!"
assert c_redseed == total, "labeled core != seed in at least one molecule!"
print("\nPASS: every generated molecule kept the frozen core intact, and the labeled core is the seed.")

## Visualize

A few grown molecules with the frozen core highlighted. We locate the core by its reserved
atom-map number (`FROZEN_MAP_BASE + label`), recovered with `frozen_core_atoms`, because
`graph_to_mol`'s canonical round-trip reorders atoms — the core is **not** at RDKit indices
`0..N-1`. The map tags are cleared for a clean depiction (same atom order), and each atom is
annotated with its original seed index.

In [ ]:
# Grid of grown molecules with the frozen core highlighted. Core atoms are located by their map
# number (rides through graph_to_mol's reordering round-trip), NOT assumed to be indices 0..N-1.
frozen_core.grown_core_grid(tagged_mols, N, save_path="check_frozen_core_grid.png")

In [ ]:
# The core is NOT at indices 0..N-1: each grown molecule's core lands wherever the canonical
# round-trip placed it -- which is exactly why we locate it by map number.
from gflownet.envs.mol_building_env import frozen_core_atoms
show = [m for m in tagged_mols if m is not None and m.GetNumAtoms() > N][:6]
print("range(N)          :", list(range(N)))
for i, m in enumerate(show):
    print(f"core idx (mol {i}) :", frozen_core_atoms(m))

In [ ]:
# Per-molecule SVGs: core highlighted (by map number) + each core atom annotated with its ORIGINAL
# seed index, wherever the canonical round-trip placed it.
from IPython.display import display
for s in frozen_core.grown_core_svgs(tagged_mols, N):
    display(s)